# Basics and Imports

In [17]:
from __future__ import print_function

import time
import requests
from pprint import pprint
import pandas as pd
import json

#host = "<SysML2-API-Repo-Host:Port>"
host = "http://localhost:9000"

# Get Projects

In [18]:
projects_url = f"{host}/projects"

try:
    print(f"Fetching projects from {projects_url}...")
    projects_response = requests.get(projects_url, timeout=5)  # 5 second timeout
    
    if projects_response.status_code == 200:
        projects = projects_response.json()
        print(f"Found {len(projects)} project(s)")
        
        # More efficient: build list instead of using concat loop
        data = [{'Project Name': p.get('name', 'N/A'), 'Project ID': p.get('@id', 'N/A')} for p in projects]
        df = pd.DataFrame(data)
        display(df)
    else:
        print(f"Error: API returned status {projects_response.status_code}")
        print(f"Response: {projects_response.text}")
        
except requests.exceptions.Timeout:
    print(f"ERROR: Request timed out. Is the API server running at {host}?")
except requests.exceptions.ConnectionError:
    print(f"ERROR: Cannot connect to {host}. Make sure the API server is running.")
except Exception as e:
    print(f"ERROR: {e}")

Fetching projects from http://localhost:9000/projects...
ERROR: Request timed out. Is the API server running at http://localhost:9000?


In [20]:

# DIAGNOSTIC: Check API connectivity and response
print("="*60)
print("API DIAGNOSTICS")
print("="*60)

test_url = f"{host}/projects"
print(f"\n1. Testing connection to: {test_url}")

try:
    response = requests.get(test_url, timeout=10)
    
    print(f"   ✓ Connected! Status Code: {response.status_code}")
    print(f"   Content-Type: {response.headers.get('Content-Type', 'N/A')}")
    print(f"   Content-Length: {len(response.text)} bytes")
    
    print(f"\n2. Raw Response (first 500 chars):")
    print(f"   {response.text[:500]}")
    
    print(f"\n3. Parsed JSON:")
    try:
        json_data = response.json()
        pprint(json_data, compact=True)
    except Exception as e:
        print(f"   Cannot parse as JSON: {e}")
        
except requests.exceptions.Timeout:
    print(f"   ✗ TIMEOUT: Server not responding within 10 seconds")
    print(f"   → Check if API is actually running")
except requests.exceptions.ConnectionError as e:
    print(f"   ✗ CONNECTION ERROR: {e}")
    print(f"   → Check if server is running on {host}")
except Exception as e:
    print(f"   ✗ ERROR: {e}")


API DIAGNOSTICS

1. Testing connection to: http://localhost:9000/projects
   ✗ TIMEOUT: Server not responding within 10 seconds
   → Check if API is actually running


In [19]:
import subprocess
import socket

print("="*70)
print("API SERVER STATUS CHECK")
print("="*70)

# 1. Check if port 9000 is listening
print("\n1. Checking if port 9000 is listening...")
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
result = sock.connect_ex(('localhost', 9000))
sock.close()

if result == 0:
    print("   ✓ Port 9000 is OPEN and listening")
else:
    print("   ✗ Port 9000 is CLOSED - API is NOT running")

# 2. Try to ping the API root
print("\n2. Testing API endpoint...")
try:
    response = requests.get("http://localhost:9000/", timeout=5)
    print(f"   ✓ API responds with status {response.status_code}")
except requests.exceptions.ConnectionError:
    print("   ✗ Cannot connect to API")
except Exception as e:
    print(f"   ✗ Error: {e}")

# 3. Check running processes (Windows netstat)
print("\n3. Checking process listening on port 9000 (netstat)...")
try:
    result = subprocess.run(
        ["netstat", "-ano", "|", "findstr", "9000"],
        capture_output=True,
        text=True,
        shell=True
    )
    if result.stdout:
        print("   ✓ Found process on port 9000:")
        print(f"   {result.stdout}")
    else:
        print("   ✗ No process found on port 9000")
except Exception as e:
    print(f"   Could not check netstat: {e}")

print("\n" + "="*70)
print("SUMMARY: If you see ✓ marks above, the API is running!")
print("="*70)


API SERVER STATUS CHECK

1. Checking if port 9000 is listening...
   ✓ Port 9000 is OPEN and listening

2. Testing API endpoint...
   ✓ API responds with status 200

3. Checking process listening on port 9000 (netstat)...
   ✓ Found process on port 9000:
     TCP    0.0.0.0:9000           0.0.0.0:0              LISTENING       26248
  TCP    127.0.0.1:14470        127.0.0.1:9000         TIME_WAIT       0
  TCP    [::]:9000              [::]:0                 LISTENING       26248
  TCP    [::1]:9000             [::1]:9371             CLOSE_WAIT      26248
  TCP    [::1]:9000             [::1]:14004            CLOSE_WAIT      26248
  TCP    [::1]:9000             [::1]:14461            CLOSE_WAIT      26248
  TCP    [::1]:9000             [::1]:14465            CLOSE_WAIT      26248
  TCP    [::1]:9000             [::1]:14471            ESTABLISHED     26248
  TCP    [::1]:9371             [::1]:9000             FIN_WAIT_2      33328
  TCP    [::1]:14004            [::1]:9000           

In [ ]:

# ADVANCED DIAGNOSTICS: Test different endpoints and response formats
print("="*70)
print("API ENDPOINT DISCOVERY")
print("="*70)

# Test different endpoints
endpoints = [
    "/",
    "/projects",
    "/api/projects",
    "/v2/projects",
    "/sysml/projects",
    "/repos",
    "/repository",
    "/model",
]

print("\nTesting various endpoints with 3-second timeout:\n")

for endpoint in endpoints:
    url = f"{host}{endpoint}"
    try:
        # Test with standard headers
        headers = {
            "Accept": "application/json",
            "Content-Type": "application/json"
        }
        response = requests.get(url, timeout=3, headers=headers)
        
        status = f"✓ {response.status_code}"
        content_len = f"({len(response.text)} bytes)"
        print(f"{endpoint:20s} {status:15s} {content_len}")
        
        # Show sample of first successful non-empty response
        if response.status_code == 200 and len(response.text) > 0:
            print(f"   Sample: {response.text[:100]}...")
            
    except requests.exceptions.Timeout:
        print(f"{endpoint:20s} ✗ TIMEOUT (3s)")
    except requests.exceptions.ConnectionError:
        print(f"{endpoint:20s} ✗ CONNECTION ERROR")
    except Exception as e:
        print(f"{endpoint:20s} ✗ {str(e)[:30]}")

print("\n" + "="*70)
print("NEXT STEP: Check which endpoint returned data")
print("="*70)

# Test /projects with longer timeout
print("\nRetrying /projects with 30-second timeout...")
try:
    response = requests.get(f"{host}/projects", timeout=30)
    print(f"Status: {response.status_code}")
    print(f"Response length: {len(response.text)} bytes")
    print(f"First 200 chars: {response.text[:200]}")
except Exception as e:
    print(f"Error: {e}")
